In [ ]:
import os
import glob
import time
import csv
import json
import tifffile
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as la
from scipy import interpolate, signal
from scipy.optimize import least_squares
import pandas as pd
import seaborn as sns
import subprocess
from functools import lru_cache

In [ ]:
# read radiometric file (csv)
def read_bfile(file):
    # ファイルのベース名に“_B.csv”を追加して新しいファイル名を作成
    fileb = os.path.splitext(file)[0] + "_B.csv"
    # CSVファイルをpandasを使って読み込む
    df = pd.read_csv(fileb)
    # パラメータ配列を初期化
    param = np.zeros((185, 5), dtype=float)
    # DataFrameからデータを抽出し、param配列に格納
    for i in range(min(185, len(df))):  # レコード数が185を超えないようにする
        for j in range(5):
            param[i, j] = float(df.iloc[i, j + 1])
    #‘CenterWavelengthNanometer’, ‘FullWidthAtHalfMaximumNanometer’,
    #‘SolarIrradianceWatt/Meter2/Micron’, ‘ReflectanceMulti’, ‘ReflectanceAdd’
    return param

# read meta data (txt)
def read_tfile(file):
    fileb = os.path.splitext(file)[0]  # ファイルの拡張子を削除し、ベース名を取得
    fileb = fileb + ".txt"  # ベース名に“.txt”を追加して、新しいファイル名を作成
    csv_file = open(fileb, "r")  # txtファイルを読み取りモードで開く
    record_list = []  # レコードを格納するリストを初期化
    record = csv_file.readline()  # 2行目から読み込みを開始
    while record :  # ファイルの終わりまでループ
        record_list.append(record.rstrip().split("="))  # 各行を読み込み、改行文字を取り除き‘=’で分割してリストにし、record_listに追加
        record = csv_file.readline()  # 次の行を読み込む
    for record in record_list:  # 全レコードをループ
        if(record[0]=="RadianceMultiVNIR                                                      "):
            radiancemultivnir = float(record[1])
        if(record[0]=="RadianceAddVNIR                                                        "):
            radianceaddvnir = float(record[1])
        if(record[0]=="RadianceMultiSWIR                                                      "):
            radiancemultiswir = float(record[1])
        if(record[0]=="RadianceAddSWIR                                                        "):
            radianceaddswir = float(record[1])
    return radiancemultivnir, radiancemultiswir, radianceaddvnir, radianceaddswir

# 入力画像に放射補正を適用
def apply_radiometric(img, radmultivnir, radmultiswir, radaddvnir, radaddswir):
    im = np.ones([img.shape[0], img.shape[1]])  # 画像の高さと幅に基づいて、全ての要素が1のマスクを作成
    # no data area
    im[img[:,:,10] == 0] = 0  # 画像のバンド10が0の位置に対して、マスクを0に設定（データがない領域を示す）
    # change to float
    img = 1.0 * img  # 入力画像を浮動小数点数型に変換
    # apply radiometric vnir
    for j in range(58):  # 0から57バンド（VNIR領域）に対して、放射補正を適用
        img[:,:,j] = img[:,:,j] * radmultivnir + radaddvnir  # 各バンドの値にradmultivnirを掛けてradaddvnirを加える
    # apply radiometric swir
    for j in range(58, 185):  # 58から184バンド（SWIR領域）に対して、放射補正を適用
        img[:,:,j] = img[:,:,j] * radmultiswir + radaddswir  # 各バンドの値にradmultiswirを掛けてradaddswirを加える
    img[im == 0] = 0  # マスクが0の位置（データがない領域）に対して、画像の値を0に設定
    return img

# ピクセル空間から地理空間への変換
def show_xy(src, x, y):
    width = src.RasterXSize # srcラスターデータセットの横幅（列数）を取得し格納
    height = src.RasterYSize # srcラスターデータセットの縦幅（行数）を取得し格納
    gt = src.GetGeoTransform() # srcラスターデータセットのジオトランスフォーム（地理変換情報）を取得し、gtに格納
    # gtは6つの要素を持つタプルで、地理座標への変換情報を含む
    # gt[0]: 左上隅のX座標（地理座標系の原点のX座標）。
    # gt[1]: 水平方向のピクセル解像度（ピクセルサイズ、X方向のスケール）。
    # gt[2]: 水平方向の回転（通常は0）。
    # gt[3]: 左上隅のY座標（地理座標系の原点のY座標）。
    # gt[4]: 垂直方向の回転（通常は0）。
    # gt[5]: 垂直方向のピクセル解像度（ピクセルサイズ、Y方向のスケール。通常は負の値、地図の上が北である場合）。
    minx = gt[0]
    miny = gt[3] + width * gt[4] + height * gt[5]
    maxx = gt[0] + width * gt[1] + height * gt[2]
    maxy = gt[3]
    X = gt[0] + x * gt[1] + y * gt[2]
    Y = gt[3] + x * gt[4] + y * gt[5]
    return X, Y

# 地理空間から緯度経度(WGS84)に変換
#def show_latlon(src, x, y):
    old_cs= osr.SpatialReference() # 元の座標系を入れるオブジェクト
    old_cs.ImportFromWkt(src.GetProjectionRef()) # データセットから取得した投影情報（WKT形式）
    # WGS84座標系のWKT（Well-Known Text）表現を文字列として定義
    wgs84_wkt = """
        GEOGCS["WGS 84",
            DATUM["WGS_1984",
                SPHEROID["WGS 84",6378137,298.257223563,
                    AUTHORITY["EPSG","7030"]],
                AUTHORITY["EPSG","6326"]],
            PRIMEM["Greenwich",0,
                AUTHORITY["EPSG","8901"]],
            UNIT["degree",0.01745329251994328,
                AUTHORITY["EPSG","9122"]],
            AUTHORITY["EPSG","4326"]]"""
    new_cs = osr.SpatialReference() # 新しい座標系を入れるオブジェクト
    new_cs .ImportFromWkt(wgs84_wkt) # 定義したWGS84のWKT文字列をインポート
    # old_csからnew_csへの座標変換を行うためのosr.CoordinateTransformationオブジェクトを作成し、transformに格納
    transform = osr.CoordinateTransformation(old_cs,new_cs)
    X, Y = show_xy(src, x, y) #ピクセル空間から地理空間への変換
    # 計算した地理座標XとYを、transformを使ってWGS84座標系（緯度経度）に変換、格納します
    latlong = transform.TransformPoint(X, Y)
    return latlong

# ハイパースペクトル画像から特定のバンドを取り出して表示するための関数
def get_rgb(img, b=3, g=14, r=29):
    ims = np.zeros([img.shape[0], img.shape[1], 3])  # 画像の高さ、幅、およびRGBの3チャンネルを持つゼロ配列を作成
    ims[:,:,0] = img[:,:,r]    #R
    ims[:,:,1] = img[:,:,g]    #G
    ims[:,:,2] = img[:,:,b]    #B
    max = np.max(ims)/3  # 画像配列の最大値を取得
    ims /= max   # 画像を max で割って正規化
    ims = np.clip(ims, 0.0, 1.0)  # 画像配列の値を0から255の範囲にクランプ
    #RGBの強さは小数点の場合0から1, 整数の場合は0から255の範囲にある必要がある。
    return ims
    
def show_img(img):
    fig, ax = plt.subplots()  # fig と ax を定義
    ax.axis("off")
    im = ax.imshow(img)  # 画像を表示
    plt.show()

# 指定した座標 (x, y) でのスペクトルデータを表示するための関数
def show_spectral(img, param, y, x): 
    fig, ax = plt.subplots()  # fig と ax を定義
    ax.plot(param[ 0: 58,0], img[y, x,  0: 58], label="VNIR Spectrum", color="r");
    ax.plot(param[58:185,0], img[y, x, 58:185], label="SWIR Spectrum", color="g");
    ax.tick_params(labelsize=16)
    ax.legend();    plt.tight_layout(); plt.show()

#SWIRのデータを抽出
def get_radiance(img, param, y, x):
    wave = param[58:185,0]
    rad = img[y, x, 58:185]
    list_data = [wave, rad]
    list_data_T = np.array(list_data).T
    return list_data_T

In [ ]:
# 範囲の指定
file = (r"E:\メタン\2025_HISUI_72_The Permian Basin-論文照合用\HSHL1G_N320W1032_20221030160051_20231127193053\HSHL1G_N320W1032_20221030160051_20231127193053.tif")
img = tifffile.imread(file) # read tif file
param = read_bfile(file)    # read radiometric file (..._B.csv)
radmultivnir, radmultiswir, radaddvnir, radaddswir = read_tfile(file)  # read meta file (....txt)
img = apply_radiometric(img, radmultivnir, radmultiswir, radaddvnir, radaddswir)  # apply radiometric parameter
ims = get_rgb(img, b=8, g=18, r=28)
center = np.array([1000, 600]) #中心座標(y, x), ここを変える
#ここで切り取りの範囲を設定
img_slice = img[center[0] - 20 : center[0] + 20, center[1] -20 : center[1] + 20, :] #補正後の画像
ims_slice = ims[center[0] - 20 : center[0] + 20, center[1] -20 : center[1] + 20, :] #画像表示用の画像
show_img(ims)
show_img(ims_slice)

In [ ]:
# H2OのLUTのあるところ
WATER_LUT_DIR = r"E:\refit\water_lut"

# read radiometric file (csv)
def read_bfile(file):
    fileb = os.path.splitext(file)[0] + "_B.csv"
    try:
        df = pd.read_csv(fileb)
    except FileNotFoundError:
        raise FileNotFoundError(f"Radiometric file not found: {fileb}")
    param = np.zeros((185, 5), dtype=float)
    for i in range(min(185, len(df))):
        for j in range(5):
            param[i, j] = float(df.iloc[i, j + 1])
    return param

# read meta data (txt)
def read_tfile(file):
    fileb = os.path.splitext(file)[0] + ".txt"
    try:
        with open(fileb, "r") as csv_file:
            record_list = [line.rstrip().split("=") for line in csv_file if "=" in line]
    except FileNotFoundError:
        raise FileNotFoundError(f"Metadata file not found: {fileb}")

    params = {}
    for record in record_list:
        if len(record) == 2:
            key = record[0].strip()
            try:
                params[key] = float(record[1])
            except ValueError:
                params[key] = record[1].strip() # Keep as string if not float

    required_keys = ["RadianceMultiVNIR", "RadianceAddVNIR", "RadianceMultiSWIR", "RadianceAddSWIR"]
    if not all(key in params for key in required_keys):
        missing = [key for key in required_keys if key not in params]
        raise ValueError(f"Missing required keys in metadata file {fileb}: {missing}")

    return params["RadianceMultiVNIR"], params["RadianceMultiSWIR"], params["RadianceAddVNIR"], params["RadianceAddSWIR"]


# 入力画像に放射補正を適用
def apply_radiometric(img, radmultivnir, radmultiswir, radaddvnir, radaddswir):
    if img.shape[2] < 185: # バンド数チェック
        raise ValueError(f"Input image has {img.shape[2]} bands, expected 185.")
    im = np.ones([img.shape[0], img.shape[1]])
    # no data area (use a band less likely to be zero, e.g., middle SWIR)
    # バンド10(index 9)が0かどうかでマスクを作成
    # HISUIのデータ仕様を確認し、適切なバンドインデックスを使用してください
    no_data_band_index = 10 # 例 (0-based index)
    if img.shape[2] > no_data_band_index:
        im[img[:,:,no_data_band_index] == 0] = 0
    else:
        print(f"Warning: Band index {no_data_band_index} for no-data mask out of bounds.")
        # Fallback: use first band maybe? Or handle differently.
        im[img[:,:,0] == 0] = 0 # Example fallback

    img_float = img.astype(float)
    # apply radiometric vnir (bands 0-57)
    img_float[:,:,:58] = img_float[:,:,:58] * radmultivnir + radianceaddvnir
    # apply radiometric swir (bands 58-184)
    img_float[:,:,58:185] = img_float[:,:,58:185] * radmultiswir + radianceaddswir
    img_float[im == 0] = 0
    return img_float

# RGB画像取得
def get_rgb(img, b=8, g=18, r=28): # デフォルトのバンド番号を元コードに合わせる
    if not (0 <= b < img.shape[2] and 0 <= g < img.shape[2] and 0 <= r < img.shape[2]):
        raise IndexError("RGB band indices are out of bounds for the image.")
    ims = np.zeros([img.shape[0], img.shape[1], 3])
    ims[:,:,0] = img[:,:,r]    #R
    ims[:,:,1] = img[:,:,g]    #G
    ims[:,:,2] = img[:,:,b]    #B
    max_val = np.max(ims)
    # 非常に暗い画像やゼロのみの画像でゼロ除算を避ける
    # また、値が小さい場合に過度に明るくなるのを防ぐため、分母に微小値を追加することも検討
    if max_val > 1e-9: # 微小値より大きい場合のみ正規化
        ims /= max_val
    ims = np.clip(ims, 0.0, 1.0)
    return ims

# 画像表示
def show_img(img):
    fig, ax = plt.subplots()
    ax.axis("off")
    im = ax.imshow(img)
    plt.show()
    plt.close(fig) # メモリ解放のために図を閉じる

# SWIRデータ抽出
def get_radiance(img, param, y, x):
    # param 配列の形状と内容を確認
    if param.shape[0] < 185 or param.shape[1] < 1:
        raise ValueError("Invalid 'param' shape for get_radiance.")
    # SWIRバンド (58-184) に対応する波長と輝度を取得
    wave = param[58:185, 0] # 1列目 (index 0) が波長
    # 画像データの形状と座標を確認
    if not (0 <= y < img.shape[0] and 0 <= x < img.shape[1]):
         raise IndexError(f"Pixel coordinates ({y},{x}) out of bounds for image shape {img.shape[:2]}")
    rad = img[y, x, 58:185] # SWIRバンドの輝度値
    list_data = [wave, rad]
    list_data_T = np.array(list_data).T
    return list_data_T # 形状: (バンド数, 2)

# メタン推定(2275~2400nm)---------------------------------------------------------------
hi_i = 57 
hi_f = 69
mo_i = 1221
mo_f = 1372

def extract_ch4(data):
    return data[:, mo_i:mo_f] 

def estimated_by_ch4(data, c): # 複数のメタン濃度レベルに対応するスペクトルデータから指定されたメタン濃度cにおけるスペクトルを線形補間で推定
    num_levels = data.shape[0] - 1 # 濃度レベルの数(今は21個)
    ch4_levels = np.arange(0, 0.25 * num_levels, 0.25)

    if c <= ch4_levels[0]: position = 1 # 0ppm以下となったら0ppmを使う
    elif c >= ch4_levels[-1]: position = len(ch4_levels) # 5ppmより多きくなったら5ppmを使う
    else: position = np.searchsorted(ch4_levels, c) # cが濃度レベルの中のどこにいるのか
    out = []
    idx_lower, idx_upper = position, position + 1 # LUTからデータを取り出す左と右
    if idx_upper >= data.shape[0]: # cが5ppmより大きかったら5ppmのを返す
        idx_upper = idx_lower = data.shape[0] - 1 # cが5ppm以上なら左も右も5ppmとする
        return data[idx_upper, :].tolist()
    elif idx_lower <= 0: # cが0ppm以下じゃないか
        idx_lower = 1 
        idx_upper = idx_lower # cが0ppm以下なら左も右も0ppmとする
        return data[idx_lower, :].tolist()

    level_lower, level_upper = ch4_levels[position-1], ch4_levels[position] # cが入っている左と右の値を取得 
    level_step = level_upper - level_lower
    r = (level_upper - c) / level_step # 線形補完の重みrの計算

    interpolated_values = r * data[idx_lower, :] + (1 - r) * data[idx_upper, :]
    return interpolated_values.tolist()


def wavelength_adjustment_ch4(data_ch4_interpolated, modtran_wavelengths, data_hisui):
    wave_width = modtran_wavelengths[1] - modtran_wavelengths[0] # 今は1nm
    data_ch4_arr = np.asarray(data_ch4_interpolated) 

    out = []
    for i in range(hi_i, hi_f): # i:HISUiのデータ行のインデックス
        hisui_wave = data_hisui.iloc[i, 0] # iの中心波長(HISUI)
        position = np.searchsorted(modtran_wavelengths, hisui_wave)
        if position == 0: out.append(data_ch4_arr[0]) # 指定したMODTRANの最小波長も小さい値を指定してたらMODTRANの一番小さいものを使う
        elif position >= len(modtran_wavelengths): out.append(data_ch4_arr[-1]) # 大きかったら
        else:
            r = (modtran_wavelengths[position] - hisui_wave) / wave_width # 線形補完の重みr
            interpolated_value = r * data_ch4_arr[position - 1] + (1 - r) * data_ch4_arr[position]
            out.append(interpolated_value)
    return out

def func_ch4(data, data_hisui, sigma, mu, a, b, c, k):
    out = instrumental_function(data, sigma, mu) # MODTRANのLUTデータに装置関数を適用
    out_extracted = extract_ch4(out) # 装置関数適用後のデータから、CH4推定に必要な波長範囲を抽出
    wavelengths_ch4_range = out_extracted[0, :] # 抽出された範囲に対応する波長データを取得
    out_reflectance_corrected = reflectance_correction(out_extracted.copy(), a, b, k) # 抽出されたデータに反射率補正を適用
    out_estimated = estimated_by_ch4(out_reflectance_corrected, c) # 反射率補正後のデータを使って、指定されたメタン濃度 c におけるスペクトルを線形補間
    out_adjusted = wavelength_adjustment_ch4(out_estimated, wavelengths_ch4_range, data_hisui) # 濃度補間された MODTRAN スペクトルを、HISUI の波長に合わせて線形補間
    return out_adjusted

def residuals_ch4(param, data, data_hisui): # 最小二乗法によって最小化される残差の計算
    sigma, mu, a, b, c, k = param
    w_est = func_ch4(data, data_hisui, sigma, mu, a, b, c, k)
    observed_rad = data_hisui.iloc[hi_i:hi_f, 1].values # HISUIデータからCH4推定に使うバンド範囲(hi_iからhi_f-1)の観測輝度をNumPy配列として取得
    w_est_np = np.asarray(w_est)
    return observed_rad - w_est_np

def estimate_param_ch4(data, data_hisui):
    max_ref = np.max(data_hisui.iloc[hi_i:hi_f, 1]) # HISUIデータで指定範囲の内最大波長のもの
    rel_max_index = np.argmax(data_hisui.iloc[hi_i:hi_f, 1]) # 輝度が最大になる相対的なインデックス
    abs_max_index = data_hisui.iloc[hi_i:hi_f].index[rel_max_index] # 絶対的なインデックスに変換
    max_wave = data_hisui.iloc[abs_max_index, 0] # 輝度が最大になるバンドの中心波長
    out = instrumental_function(data.copy(), sigma=6.5, mu=0.0) # CH4LUTデータに固定パラメータ(sigma=6.5, mu=0.0)を使って装置関数を適用
    out_extracted = extract_ch4(out) # 装置関数適用後のデータからCH4関連の波長範囲を抽出
    out_estimated = estimated_by_ch4(out_extracted, c=1.8) # 抽出されたデータに対し固定のCH4濃度c=1.8で線形補間
    modtran_wavelengths_ch4_range = out_extracted[0, :] # 抽出された範囲の波長データを取得
    out_adjusted = wavelength_adjustment_ch4(out_estimated, modtran_wavelengths_ch4_range, data_hisui)
    modtran_val_at_max = out_adjusted[rel_max_index]

    b = max_ref / max_wave / modtran_val_at_max
    a = 0.0
    return b, a


# H2O推定----------------------------------------------------------------
h2o_band_i = 11
h2o_band_f = 28
h2o_modtran_i = 645
h2o_modtran_f = 865

def extract_h2o(data):
    return data[:, h2o_modtran_i:h2o_modtran_f]

def estimated_by_water(data, w):
    num_levels = data.shape[0] - 1
    water_levels = np.arange(0, 0.25 * num_levels, 0.25)
    if w <= water_levels[0]: position = 1
    elif w >= water_levels[-1]: position = len(water_levels)
    else: position = np.searchsorted(water_levels, w)
    out = []
    idx_lower, idx_upper = position, position + 1
    if idx_upper >= data.shape[0]:
        idx_upper = idx_lower = data.shape[0] - 1
        return data[idx_upper, :].tolist()
    elif idx_lower <= 1: 
        idx_lower = 1
        idx_upper = idx_lower
        return data[idx_lower, :].tolist()
    level_lower, level_upper = water_levels[position-1], water_levels[position]
    level_step = level_upper - level_lower
    r = (level_upper - w) / level_step if level_step > 0 else 0.0
    interpolated_values = r * data[idx_lower, :] + (1 - r) * data[idx_upper, :]
    return interpolated_values.tolist()

def wavelength_adjustment_h2o(data_water_interpolated, modtran_wavelengths, data_hisui):
    wave_width = modtran_wavelengths[1] - modtran_wavelengths[0]
    data_water_arr = np.asarray(data_water_interpolated)
    out = []
    for i in range(h2o_band_i, h2o_band_f):
        hisui_wave = data_hisui.iloc[i, 0]
        position = np.searchsorted(modtran_wavelengths, hisui_wave)
        if position == 0: out.append(data_water_arr[0])
        elif position >= len(modtran_wavelengths): out.append(data_water_arr[-1])
        else:
            r = (modtran_wavelengths[position] - hisui_wave) / wave_width
            interpolated_value = r * data_water_arr[position - 1] + (1 - r) * data_water_arr[position]
            out.append(interpolated_value)
    return out

def func_h2o(data, data_hisui, sigma, mu, a, b, w, k):
    out = instrumental_function(data, sigma, mu)
    out_extracted = extract_h2o(out)
    wavelengths_h2o_range = out_extracted[0, :] 
    out_reflectance_corrected = reflectance_correction(out_extracted.copy(), a, b, k)
    out_estimated = estimated_by_water(out_reflectance_corrected, w)
    out_adjusted = wavelength_adjustment_h2o(out_estimated, wavelengths_h2o_range, data_hisui)
    return out_adjusted

def residuals_h2o(param, data, data_hisui):
    sigma, mu, a, b, w, k = param
    w_est = func_h2o(data, data_hisui, sigma, mu, a, b, w, k)
    observed_rad = data_hisui.iloc[h2o_band_i:h2o_band_f, 1].values
    w_est_np = np.asarray(w_est)

def estimate_param_h2o(data, data_hisui):
    out = instrumental_function(data.copy(), sigma=6.5, mu=0.0)
    out_extracted = extract_h2o(out)
    out_estimated = estimated_by_water(out_extracted, w=2.0)
    modtran_wavelengths_h2o_range = out_extracted[0, :] 
    out_adjusted = wavelength_adjustment_h2o(out_estimated, modtran_wavelengths_h2o_range, data_hisui)
    out_np = np.array(out_adjusted)
    highlight_points = [0, 1, 2, 15, 16]
    hisui_subset = data_hisui.iloc[h2o_band_i:h2o_band_f]
    max_highlight_idx = max(highlight_points)
    x_waves = hisui_subset.iloc[highlight_points, 0].values
    y_observed = hisui_subset.iloc[highlight_points, 1].values
    y_modtran = out_np[highlight_points]
    y = y_observed / y_modtran
    coefficients = np.polyfit(x_waves, y, 1)
    return coefficients[0], coefficients[1] 
    
def instrumental_function(data, sigma, mu):
    wavelengths = data[0, :]
    num_levels = data.shape[0] - 1
    out = np.zeros_like(data)
    out[0, :] = wavelengths
    ave_wave = np.mean(wavelengths)
    sigma = max(sigma, 1e-6)
    gauss = np.exp(-(wavelengths - ave_wave - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
    for i in range(1, num_levels + 1):
        y = data[i, :]
        out[i, :] = np.convolve(y, gauss, mode='same')
    return out

def reflectance_correction(data, a, b, k):
    data_corrected = data.copy()
    wavelengths = data_corrected[0, :]
    num_levels = data_corrected.shape[0] - 1
    for i in range(1, num_levels + 1):
        reflectance_factor = a + b * wavelengths
        reflectance_factor = np.maximum(reflectance_factor, 0) 
        data_corrected[i, :] = data_corrected[i, :] * reflectance_factor + k
    return data_corrected

def list_available_h2o_values(dir_path: str): # 水蒸気量とファイルパスを整理
    paths = glob.glob(os.path.join(dir_path, "*.csv")) # 拡張子が.csvのやつを取得
    table = []
    for p in paths:
        name = os.path.splitext(os.path.basename(p))[0] # ファイルパスpからファイル名部分だけを取り出し(os.path.basename)拡張子を除いた部分を取得→ファイル名が水蒸気量になっているため
        name_std = name.replace(",", ".")
        h2o = float(name_std)
        table.append((h2o, p)) # h20:水蒸気量,p:ファイルパス
    table.sort(key=lambda t: t[0]) # 各タプルの最初の要素（水蒸気量の数値t[0]）に基づいて小さい順に並べ替え
    h2o_values = np.array([t[0] for t in table], dtype=float) # 水蒸気量のみのリスト
    file_paths = [t[1] for t in table]
    return h2o_values, file_paths

def idx_for_h2o(h: float) -> int: # 推定された水蒸気量hを入力とし利用可能なLUTの水蒸気量リストH2O_VALUESの中でhに最も近い値を持つLUTのインデックス(H2O_VALUES における位置番号)を取り出す
    if len(H2O_VALUES) == 1: return 0 # 利用可能なLUTが一つしかない場合そのインデックスを返す
    insert_idx = np.searchsorted(H2O_VALUES, h)
    if insert_idx == 0: return 0 # hが最小値以下の場合最小値のインデックスを返す
    if insert_idx == len(H2O_VALUES): return len(H2O_VALUES) - 1 # hが最大値以上の場合最大値のインデックスを返す

    dist_left = h - H2O_VALUES[insert_idx - 1] # hと左右の差
    dist_right = H2O_VALUES[insert_idx] - h
    return insert_idx - 1 if dist_left < dist_right else insert_idx # 左右の差が小さい方を返す


def idx_for_h2o_sticky(h: float, last_idx: int, margin: float = 0.02) -> int: # スティッキー選択 gemini製
    if len(H2O_VALUES) == 0:
      raise ValueError("No H2O LUT values available for sticky selection.")
    if len(H2O_VALUES) == 1: return 0
    if last_idx is None: return idx_for_h2o(h)

    lower_mid = -np.inf if last_idx == 0 else (H2O_VALUES[last_idx-1] + H2O_VALUES[last_idx]) / 2.0
    upper_mid = np.inf if last_idx == len(H2O_VALUES) - 1 else (H2O_VALUES[last_idx] + H2O_VALUES[last_idx+1]) / 2.0
    lower_bound = lower_mid - margin
    upper_bound = upper_mid + margin
    if lower_bound <= h < upper_bound: return last_idx
    return idx_for_h2o(h)


H2O_VALUES, H2O_FILES = list_available_h2o_values(WATER_LUT_DIR)
if len(H2O_VALUES) > 1:
    MIDPOINTS = (H2O_VALUES[:-1] + H2O_VALUES[1:]) / 2.0
else:
    MIDPOINTS = np.array([])
print(f"Found {len(H2O_VALUES)} CH4 LUTs for H2O values: {H2O_VALUES}")


# LUT読み込み------------------------------------------------------
@lru_cache(maxsize=64)
def load_ch4_lut(filepath: str):
    df = pd.read_csv(filepath, header=0)
    data_ch4_raw = df.to_numpy(dtype=object)
    wavelengths = data_ch4_raw[:, 0].astype(float)
    ch4_data = data_ch4_raw[:, 1:].astype(float)
    data_ch4_raw = np.hstack((wavelengths[:, np.newaxis], ch4_data))
    data_ch4 = data_ch4_raw.T
    if data_ch4.shape[0] > 1:
        data_ch4[1:, :] = data_ch4[1:, :] * 100 # スケーリング
    wavelength_row = data_ch4[0, :]
    return data_ch4

# 
def estimate_h2o_and_ch4_per_pixel( # 各ピクセルに対して水蒸気量を推定しその結果に基づいて適切なメタンLUTを選択し最終的にメタンのパラメータを推定する
    img_slice: np.ndarray,
    param,
    data_h2o, # H2O LUT
    sticky: bool = True,
    sticky_margin: float = 0.02,
    ch4_init_ppm: float = 1.8,
):
    H, W, C = img_slice.shape
    out_h2o = np.full((H, W, 1), np.nan, dtype=float)
    out_ch4 = np.full((H, W, 4), np.nan, dtype=float)
    start_time = time.time()
    processed_pixels = 0 # 処理済みのピクセル数
    error_pixels_h2o = 0
    error_pixels_lut = 0
    error_pixels_ch4 = 0
    for y in range(H):
        last_ch4_lut_idx = None # スティッキー LUT 選択で使用する、「前のピクセルで選択した LUT のインデックス」をリセット
        row_start_time = time.time()
        for x in range(W):

            # h2o推定
            data_rad = get_radiance(img_slice, param, y, x) # 処理中のピクセルのSWIR輝度スペクトルデータを取得
            df_hisui = pd.DataFrame(data_rad, columns=["Wavelength", "Radiance"]) # pandas DataFrameに変換
            if np.max(df_hisui["Radiance"]) <= 0: # 輝度が0のピクセルはスキップ
                out_h2o[y, x, 0] = 0.0
                out_ch4[y, x, :] = 0.0
                processed_pixels += 1
                continue
            b_h2o, a_h2o = estimate_param_h2o(data_h2o, df_hisui) # a,bの初期値の推定
            a0_h2o = np.array([6.5, 0.0, a_h2o, b_h2o, 2.0, 0.0]) # H2Oの初期推定値はここで変える
            res_h2o = least_squares(residuals_h2o, a0_h2o, args=(data_h2o, df_hisui), method="lm")
            h2o_est = float(res_h2o.x[4])
            h2o_est = max(0, h2o_est)
            out_h2o[y, x, 0] = h2o_est

            # 推定したh2oから適切なLUTを選ぶ
            if sticky:
                idx = idx_for_h2o_sticky(h2o_est, last_ch4_lut_idx, margin=sticky_margin)
            else:
                idx = idx_for_h2o(h2o_est)
            last_ch4_lut_idx = idx # 選択したインデックスを記憶
            selected_ch4_lut_file = H2O_FILES[idx]
            data_ch4 = load_ch4_lut(selected_ch4_lut_file) # 選択されたファイルパスを使ってCH4 LUTデータを読み込み

            # ch4推定
            b_ch4_init, a_ch4_init = estimate_param_ch4(data_ch4, df_hisui) # a,bの初期値の推定
            a0_ch4 = np.array([6.0, 0.0, a_ch4_init, b_ch4_init, ch4_init_ppm, 0.0]) # ch4の初期値は後で指定
            res_ch4 = least_squares(residuals_ch4, a0_ch4, args=(data_ch4, df_hisui), method="lm")
            out_ch4[y, x, 0] = float(res_ch4.x[0])  # sigma
            out_ch4[y, x, 1] = float(res_ch4.x[1])  # mu
            ch4_ppm_est = float(res_ch4.x[4])
            ch4_ppm_est = max(0, ch4_ppm_est)
            out_ch4[y, x, 2] = ch4_ppm_est # ch4
            residual_values = residuals_ch4(res_ch4.x, data_ch4, df_hisui)
            out_ch4[y, x, 3] = float(np.sum(residual_values**2)) # "E:\refit\estimated_ch4_residual.csv"に保存
    end_time = time.time()
    total_pixels = H * W
    print(f"\n--- Processing Summary ---")
    print(f"Total pixels processed: {processed_pixels}/{total_pixels}")
    print(f"Errors during H2O estimation: {error_pixels_h2o}")
    print(f"Errors during LUT selection/load: {error_pixels_lut}")
    print(f"Errors during CH4 estimation: {error_pixels_ch4}")
    print(f"Total NaN in H2O results: {np.isnan(out_h2o).sum()}")
    print(f"Total NaN in CH4 results (first channel): {np.isnan(out_ch4[:,:,0]).sum()}")
    print(f"Total processing time: {end_time - start_time:.2f} seconds.")
    return out_h2o, out_ch4

# 

h2o_lut_file = r"E:\refit\waterprox\modtranwaterdata_full.csv" 
data_h2o_df = pd.read_csv(h2o_lut_file)
data_h2o_np = data_h2o_df.to_numpy().T
    
if data_h2o_np.shape[0] > 1: # スケーリング適用
    data_h2o_np[1:, :] = data_h2o_np[1:, :] * 100
print(f"H2O LUT loaded from '{h2o_lut_file}' and prepared.")
print(f"H2O LUT shape: {data_h2o_np.shape}")
if 'img_slice' in locals() and isinstance(img_slice, np.ndarray) \
    and 'param' in locals() and isinstance(param, np.ndarray):
    print(f"img_slice shape: {img_slice.shape}")
    out_h2o, out_ch4 = estimate_h2o_and_ch4_per_pixel(img_slice, param, data_h2o_np,sticky=True,sticky_margin=0.02,ch4_init_ppm=1.8)
    # 結果表示
    print("\n--- Results ---")
    print("Estimated H2O (sample):\n", out_h2o[::2, ::2, 0])
    print("\nEstimated CH4 ppm (sample):\n", out_ch4[::2, ::2, 2])
    # 結果をCSVファイルに保存
    h, w, _ = out_h2o.shape
    h2o_2d = out_h2o.reshape(h, w)
    df_h2o = pd.DataFrame(h2o_2d)
    df_h2o.to_csv("estimated_h2o.csv", index=False, header=False, na_rep='NaN')
    print("\nEstimated H2O saved to estimated_h2o.csv")

    ch4_sigma_2d = out_ch4[:, :, 0]
    ch4_mu_2d = out_ch4[:, :, 1]
    ch4_ppm_2d = out_ch4[:, :, 2]
    ch4_res_2d = out_ch4[:, :, 3]
    df_ch4_sigma = pd.DataFrame(ch4_sigma_2d)
    df_ch4_mu = pd.DataFrame(ch4_mu_2d)
    df_ch4_ppm = pd.DataFrame(ch4_ppm_2d)
    df_ch4_res = pd.DataFrame(ch4_res_2d)
    df_ch4_sigma.to_csv("estimated_ch4_sigma.csv", index=False, header=False, na_rep='NaN')
    df_ch4_mu.to_csv("estimated_ch4_mu.csv", index=False, header=False, na_rep='NaN')
    df_ch4_ppm.to_csv("estimated_ch4_ppm.csv", index=False, header=False, na_rep='NaN')
    df_ch4_res.to_csv("estimated_ch4_residual.csv", index=False, header=False, na_rep='NaN')
    print("Estimated CH4 parameters saved to separate CSV files.")

else:
    print("Error: img_slice or param not defined or not NumPy arrays.")
    if 'img_slice' not in locals(): print(" - img_slice is missing.")
    elif not isinstance(img_slice, np.ndarray): print(f" - img_slice is type {type(img_slice)}, expected NumPy array.")
    if 'param' not in locals(): print(" - param is missing.")
    elif not isinstance(param, np.ndarray): print(f" - param is type {type(param)}, expected NumPy array.")
